# LeetCode 1052 — Grumpy Bookstore Owner bằng Python

Link đề: https://leetcode.com/problems/grumpy-bookstore-owner/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Sliding Window (cửa sổ trượt)

Skeleton LeetCode:

```python
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        ...
```

Ghi chú: một chủ hiệu sách mở cửa `n` phút; phút `i` có `customers[i]` khách vào. Những phút chủ quán **cáu kỉnh** (`grumpy[i] = 1`) thì khách **không hài lòng**, các phút còn lại thì hài lòng. Chủ quán có một "bí kíp" giúp **không cáu** trong `minutes` phút **liên tiếp**, nhưng chỉ dùng **một lần**. Tìm số khách hài lòng **nhiều nhất**. Quan sát then chốt: dùng bí kíp ở đoạn nào thì **thêm** những khách của các phút cáu trong đoạn đó (các phút không cáu trong đoạn vốn đã hài lòng); vậy đáp án là `(khách hài lòng sẵn có) + (tổng khách các phút cáu nhiều nhất trong một cửa sổ độ dài minutes)`. Đây là bài **cửa sổ trượt** độ dài cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và **tách bài toán** thành "phần chắc chắn có" (khách của phút không cáu) cộng "phần thêm được" (khách phút cáu trong cửa sổ bí kíp).
2. Nắm các công cụ Python cần thiết: `zip`, tích từng phần tử `c * g`, `itertools.accumulate` (tổng tiền tố), cửa sổ trượt cố định bằng cách cộng phần tử vào và trừ phần tử ra, và `max`.
3. Hiểu nguyên lý **cửa sổ trượt độ dài cố định**: khi dịch cửa sổ một phút, cộng phút mới vào và trừ phút cũ ra: `O(1)` mỗi bước.
4. Đi qua 4 cách: thử từng vị trí bắt đầu bí kíp và mô phỏng lại cả ngày (brute force), dùng hai mảng tổng tiền tố (hài lòng sẵn và tổng khách), cửa sổ trượt trên "khách bị mất" (cách nộp chính), và dùng `accumulate` trên mảng "khách bị mất".
5. Tránh các lỗi kinh điển: quên trừ phần tử rời cửa sổ, bỏ qua cửa sổ đầu tiên, tính **cả khách hài lòng sẵn** trong cửa sổ (đếm đôi), lệch một đơn vị độ dài cửa sổ, tham lam chọn phút cáu đông nhất, và quên cộng phần hài lòng sẵn có.

## 2. Đề bài & ví dụ

Một chủ hiệu sách mở cửa trong `n` phút. Cho mảng nguyên `customers` độ dài `n`, `customers[i]` là số khách vào hiệu sách vào **đầu phút** thứ `i` và tất cả rời đi sau **cuối phút đó**.

Trong một số phút chủ quán **cáu kỉnh**. Cho mảng nhị phân `grumpy`: `grumpy[i] = 1` nếu chủ quán cáu trong phút thứ `i`, ngược lại là `0`.

Khi chủ quán cáu, khách vào phút đó **không hài lòng**; ngược lại họ **hài lòng**.

Chủ quán biết một bí kíp để **không cáu** trong `minutes` phút **liên tiếp**, nhưng chỉ dùng được **một lần**.

Trả về **số khách hài lòng nhiều nhất** có thể trong cả ngày.

**Ví dụ 1:** `customers = [1,0,1,2,1,1,7,5]`, `grumpy = [0,1,0,1,0,1,0,1]`, `minutes = 3` -> `16`. Dùng bí kíp ở ba phút cuối: `1 + 1 + 1 + 1 + 7 + 5 = 16`.

**Ví dụ 2:** `customers = [1]`, `grumpy = [0]`, `minutes = 1` -> `1`.

**Ràng buộc:**

- `n == customers.length == grumpy.length`
- `1 <= minutes <= n <= 2 * 10^4`
- `0 <= customers[i] <= 1000`
- `grumpy[i]` là `0` hoặc `1`.

In [1]:
from typing import List, Tuple
from itertools import accumulate
import random, time

examples = [
    ([1, 0, 1, 2, 1, 1, 7, 5], [0, 1, 0, 1, 0, 1, 0, 1], 3, 16),
    ([1], [0], 1, 1),
]
for customers, grumpy, minutes, expected in examples:
    print(f"customers={customers} grumpy={grumpy} minutes={minutes} -> kỳ vọng {expected}")

customers=[1, 0, 1, 2, 1, 1, 7, 5] grumpy=[0, 1, 0, 1, 0, 1, 0, 1] minutes=3 -> kỳ vọng 16
customers=[1] grumpy=[0] minutes=1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tách "có sẵn" và "thêm được"

Chia tổng số khách hài lòng thành hai phần:

- **Có sẵn:** `base = sum(customers[i] for i where grumpy[i] == 0)`; các phút không cáu luôn hài lòng, dù ta có dùng bí kíp ở đâu.
- **Thêm được:** nếu dùng bí kíp ở cửa sổ `[s, s + minutes - 1]`, các phút **cáu** trong cửa sổ trở thành hài lòng: thêm `sum(customers[i] for i in cửa sổ if grumpy[i] == 1)`. (Các phút không cáu trong cửa sổ vốn đã nằm trong `base`, không được tính hai lần.)

In [2]:
customers, grumpy, minutes = examples[0][:3]
base = sum(c for c, g in zip(customers, grumpy) if g == 0)
lost = [c * g for c, g in zip(customers, grumpy)]      # khách bị mất vì cáu (0 nếu không cáu)
print("base =", base, "| lost =", lost)
for s in range(len(customers) - minutes + 1):
    print(f"cửa sổ bắt đầu {s}: thêm được {sum(lost[s:s + minutes])} -> tổng {base + sum(lost[s:s + minutes])}")

base = 10 | lost = [0, 0, 0, 2, 0, 1, 0, 5]
cửa sổ bắt đầu 0: thêm được 0 -> tổng 10
cửa sổ bắt đầu 1: thêm được 2 -> tổng 12
cửa sổ bắt đầu 2: thêm được 2 -> tổng 12
cửa sổ bắt đầu 3: thêm được 3 -> tổng 13
cửa sổ bắt đầu 4: thêm được 1 -> tổng 11
cửa sổ bắt đầu 5: thêm được 6 -> tổng 16


### 3.2. Cửa sổ trượt độ dài cố định

Gọi `lost[i] = customers[i] * grumpy[i]`. Ta cần `max` của tổng `lost` trên mọi cửa sổ độ dài `minutes`. Tính tổng cửa sổ đầu tiên rồi **dịch** từng bước: `window += lost[i] - lost[i - minutes]` (cộng phút mới vào, trừ phút vừa rời). Mỗi bước `O(1)`, tổng `O(n)`.

In [3]:
lost = [1 * 0, 0 * 1, 1 * 0, 2 * 1, 1 * 0, 1 * 1, 7 * 0, 5 * 1]
minutes = 3
window = sum(lost[:minutes])
best = window
for i in range(minutes, len(lost)):
    window += lost[i] - lost[i - minutes]
    best = max(best, window)
    print(f"i={i}: cộng {lost[i]}, trừ {lost[i - minutes]} -> window = {window}, best = {best}")

i=3: cộng 2, trừ 0 -> window = 2, best = 2
i=4: cộng 0, trừ 0 -> window = 2, best = 2
i=5: cộng 1, trừ 0 -> window = 3, best = 3
i=6: cộng 0, trừ 2 -> window = 1, best = 3
i=7: cộng 5, trừ 0 -> window = 6, best = 6


### 3.3. Tổng tiền tố

`pre[i] = lost[0] + ... + lost[i - 1]` cho phép tính tổng cửa sổ `[l, r)` bằng `pre[r] - pre[l]` trong `O(1)`. `itertools.accumulate(lost, initial=0)` tạo `pre` có độ dài `n + 1`.

In [4]:
pre = list(accumulate(lost, initial=0))
print("pre =", pre, "| tổng lost[5:8] =", pre[8] - pre[5], "=", sum(lost[5:8]))

pre = [0, 0, 0, 0, 2, 2, 3, 3, 8] | tổng lost[5:8] = 6 = 6


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `customers = [1,0,1,2,1,1,7,5]`, `grumpy = [0,1,0,1,0,1,0,1]`, `minutes = 3`.

- Có sẵn (phút không cáu): `1 + 1 + 1 + 7 = 10` (các phút `0, 2, 4, 6`).
- Khách bị mất vì cáu `lost = [0, 0, 0, 2, 0, 1, 0, 5]`.
- Tổng `lost` mọi cửa sổ độ dài `3`: `[0, 2, 2, 3, 1, 6]`; lớn nhất `6` (cửa sổ phút `5..7`: `1 + 0 + 5`).
- Đáp án `10 + 6 = 16` ✓.

Ví dụ 2: `[1]`, không cáu: có sẵn `1`, `lost = [0]`, thêm `0`: đáp án `1`.

## 5. Cách 1 — Brute Force: thử từng vị trí bắt đầu và mô phỏng cả ngày

**Ý tưởng.** Với mỗi vị trí bắt đầu `s` của bí kíp (`0 <= s <= n - minutes`), tạo mảng `grumpy'` bằng cách **đặt về `0`** các phút trong `[s, s + minutes)`, rồi mô phỏng cả ngày: cộng `customers[i]` ở những phút có `grumpy'[i] == 0`. Lấy tối đa. Đúng định nghĩa của đề (không dùng phân tích "có sẵn" và "thêm") nên là **oracle độc lập**.

In [5]:
def satisfied_brute(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    best = 0
    for s in range(n - minutes + 1):
        g = grumpy[:]
        for i in range(s, s + minutes):
            g[i] = 0                                    # dùng bí kíp: không cáu trong cửa sổ
        best = max(best, sum(c for c, gi in zip(customers, g) if gi == 0))
    return best

for customers, grumpy, minutes, expected in examples:
    got = satisfied_brute(customers, grumpy, minutes)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 16 OK
brute = 1 OK


**Độ phức tạp và nút thắt.** `O(n * n)` (mỗi vị trí bắt đầu mô phỏng `O(n)`): với `n = 2 * 10^4` là `~ 4 * 10^8` phép: quá chậm. Đo thử:

In [6]:
rng = random.Random(1052)
for n in (500, 1000, 2000, 4000):
    customers = [rng.randint(0, 1000) for _ in range(n)]
    grumpy = [rng.randint(0, 1) for _ in range(n)]
    t0 = time.perf_counter()
    satisfied_brute(customers, grumpy, n // 4)
    dt = time.perf_counter() - t0
    print(f"n={n:5}: {dt * 1000:9.1f} ms -> ngoại suy n=2*10^4: ~{dt * (2 * 10 ** 4 / n) ** 2:8.1f} s")

n=  500:      15.6 ms -> ngoại suy n=2*10^4: ~    25.0 s
n= 1000:      56.8 ms -> ngoại suy n=2*10^4: ~    22.7 s


n= 2000:     166.0 ms -> ngoại suy n=2*10^4: ~    16.6 s


n= 4000:     686.5 ms -> ngoại suy n=2*10^4: ~    17.2 s


Nút thắt: mỗi vị trí bắt đầu ta tính lại **cả ngày**, trong khi các phút ngoài cửa sổ không đổi. Ta tách "có sẵn" và chỉ tính phần **thêm** do cửa sổ.

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án `= base + max_s sum(lost[s : s + minutes])`.**

Với bí kíp ở cửa sổ `W`: số khách hài lòng `= sum(customers[i] : i không cáu) + sum(customers[i] : i cáu và i thuộc W)` vì các phút không cáu trong `W` đã được tính ở số hạng đầu. Số hạng đầu là `base` không đổi; số hạng thứ hai là tổng `lost` trong `W`. Do đó chỉ cần **tổng `lost` lớn nhất trên cửa sổ độ dài `minutes`**: cửa sổ trượt `O(n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Hai mảng tổng tiền tố: `satisfied` (hài lòng sẵn) và `total` (mọi khách): đáp án tại `s` = `sat[s] + (total[s + m] - total[s]) + (sat[n] - sat[s + m])` |
| 3 | Cửa sổ trượt trên `lost` (nộp) |
| 4 | Tổng tiền tố của `lost` bằng `accumulate`, lấy `max` các hiệu cách nhau `minutes` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai mảng tổng tiền tố

**Giải thích từng đoạn.**

- `sat[i]` = số khách hài lòng sẵn trong `i` phút đầu (`customers[j]` nếu `grumpy[j] == 0`).
- `tot[i]` = tổng số khách trong `i` phút đầu.
- Dùng bí kíp ở `[s, s + m)`: hài lòng = `sat[s]` (trước cửa sổ) + `tot[s + m] - tot[s]` (cả cửa sổ đều hài lòng) + `sat[n] - sat[s + m]` (sau cửa sổ).
- Lấy `max` theo `s`.

In [7]:
def satisfied_prefix(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    sat = list(accumulate((c if g == 0 else 0 for c, g in zip(customers, grumpy)), initial=0))
    tot = list(accumulate(customers, initial=0))
    return max(sat[s] + (tot[s + minutes] - tot[s]) + (sat[n] - sat[s + minutes]) for s in range(n - minutes + 1))

for customers, grumpy, minutes, expected in examples:
    got = satisfied_prefix(customers, grumpy, minutes)
    print(f"tiền tố = {got}", "OK" if got == expected else "SAI")
    assert got == expected

tiền tố = 16 OK
tiền tố = 1 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        from itertools import accumulate
        n = len(customers)
        sat = list(accumulate((c if g == 0 else 0 for c, g in zip(customers, grumpy)), initial=0))   # hài lòng sẵn
        tot = list(accumulate(customers, initial=0))                                                  # mọi khách
        return max(sat[s] + (tot[s + minutes] - tot[s]) + (sat[n] - sat[s + minutes])
                   for s in range(n - minutes + 1))

sol_prefix = Solution().maxSatisfied
for customers, grumpy, minutes, expected in examples:
    assert sol_prefix(customers, grumpy, minutes) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(n)` cho hai mảng tiền tố.

---

### 7.2. Cách 3 — Cửa sổ trượt trên `lost` (cách nộp chính)

**Giải thích từng đoạn.**

- `base`: tổng khách ở phút không cáu.
- `lost[i] = customers[i] * grumpy[i]`.
- `window`: tổng `lost` trong cửa sổ hiện tại; ban đầu `sum(lost[:minutes])`.
- Với `i` từ `minutes` đến `n - 1`: `window += lost[i] - lost[i - minutes]`; cập nhật `best`.
- Đáp án `base + best`.

In [9]:
def satisfied_window(customers: List[int], grumpy: List[int], minutes: int) -> int:
    base = sum(c for c, g in zip(customers, grumpy) if g == 0)
    lost = [c * g for c, g in zip(customers, grumpy)]
    window = sum(lost[:minutes])
    best = window
    for i in range(minutes, len(lost)):
        window += lost[i] - lost[i - minutes]
        best = max(best, window)
    return base + best

for customers, grumpy, minutes, expected in examples:
    got = satisfied_window(customers, grumpy, minutes)
    print(f"cửa sổ trượt = {got}", "OK" if got == expected else "SAI")
    assert got == expected

cửa sổ trượt = 16 OK
cửa sổ trượt = 1 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        n = len(customers)
        base = 0                                   # khách hài lòng sẵn (phút không cáu)
        for i in range(n):
            if grumpy[i] == 0:
                base += customers[i]
        window = 0                                 # tổng khách bị mất trong cửa sổ hiện tại
        for i in range(minutes):
            window += customers[i] * grumpy[i]
        best = window
        for i in range(minutes, n):
            window += customers[i] * grumpy[i] - customers[i - minutes] * grumpy[i - minutes]   # vào một phút, ra một phút
            best = max(best, window)
        return base + best

sol_window = Solution().maxSatisfied
for customers, grumpy, minutes, expected in examples:
    assert sol_window(customers, grumpy, minutes) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(1)` thêm.

---

### 7.3. Cách 4 — Tổng tiền tố của `lost` bằng `accumulate`

**Ý tưởng khác.** Với `pre = accumulate(lost, initial=0)`, tổng cửa sổ `[s, s + m)` là `pre[s + m] - pre[s]`. Lấy `max` của hiệu `pre[i + m] - pre[i]` theo `i` rồi cộng `base`. Viết gọn bằng một biểu thức.

In [11]:
def satisfied_accumulate(customers: List[int], grumpy: List[int], minutes: int) -> int:
    lost = [c * g for c, g in zip(customers, grumpy)]
    pre = list(accumulate(lost, initial=0))
    base = sum(customers) - pre[-1]                  # mọi khách trừ khách bị mất = khách hài lòng sẵn
    return base + max(pre[i + minutes] - pre[i] for i in range(len(lost) - minutes + 1))

for customers, grumpy, minutes, expected in examples:
    got = satisfied_accumulate(customers, grumpy, minutes)
    print(f"accumulate = {got}", "OK" if got == expected else "SAI")
    assert got == expected

accumulate = 16 OK
accumulate = 1 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        from itertools import accumulate
        lost = [c * g for c, g in zip(customers, grumpy)]      # khách bị mất vì cáu
        pre = list(accumulate(lost, initial=0))
        base = sum(customers) - pre[-1]                         # khách hài lòng sẵn = mọi khách - khách bị mất
        return base + max(pre[i + minutes] - pre[i] for i in range(len(lost) - minutes + 1))

sol_accumulate = Solution().maxSatisfied
for customers, grumpy, minutes, expected in examples:
    assert sol_accumulate(customers, grumpy, minutes) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cửa sổ trượt cho ví dụ 1

In `window` và `best` ở mỗi bước (kèm cửa sổ hiện tại).

In [13]:
def trace_window(customers, grumpy, minutes):
    base = sum(c for c, g in zip(customers, grumpy) if g == 0)
    lost = [c * g for c, g in zip(customers, grumpy)]
    print("base =", base, "| lost =", lost)
    window = sum(lost[:minutes])
    best = window
    print(f"cửa sổ đầu [0, {minutes}): window = {window}")
    for i in range(minutes, len(lost)):
        window += lost[i] - lost[i - minutes]
        best = max(best, window)
        print(f"i={i}: cửa sổ [{i - minutes + 1}, {i + 1}) = {lost[i - minutes + 1:i + 1]} -> window = {window}, best = {best}")
    print(f"đáp án = base + best = {base} + {best} = {base + best}")
    return base + best

assert trace_window([1, 0, 1, 2, 1, 1, 7, 5], [0, 1, 0, 1, 0, 1, 0, 1], 3) == 16

base = 10 | lost = [0, 0, 0, 2, 0, 1, 0, 5]
cửa sổ đầu [0, 3): window = 0
i=3: cửa sổ [1, 4) = [0, 0, 2] -> window = 2, best = 2
i=4: cửa sổ [2, 5) = [0, 2, 0] -> window = 2, best = 2
i=5: cửa sổ [3, 6) = [2, 0, 1] -> window = 3, best = 3
i=6: cửa sổ [4, 7) = [0, 1, 0] -> window = 1, best = 3
i=7: cửa sổ [5, 8) = [1, 0, 5] -> window = 6, best = 6
đáp án = base + best = 10 + 6 = 16


### 8.2. Tổng tiền tố cho ví dụ 1

In `pre` của `lost` và các hiệu `pre[i + m] - pre[i]`.

In [14]:
customers, grumpy, minutes = examples[0][:3]
lost = [c * g for c, g in zip(customers, grumpy)]
pre = list(accumulate(lost, initial=0))
print("pre =", pre)
diffs = [pre[i + minutes] - pre[i] for i in range(len(lost) - minutes + 1)]
print("tổng cửa sổ độ dài 3:", diffs, "| lớn nhất:", max(diffs))
assert max(diffs) == 6

pre = [0, 0, 0, 0, 2, 2, 3, 3, 8]
tổng cửa sổ độ dài 3: [0, 2, 2, 3, 1, 6] | lớn nhất: 6


### 8.3. Vì sao không được tính cả khách "đã hài lòng" trong cửa sổ

Nếu dùng tổng `customers` trong cửa sổ (thay vì `lost`) rồi cộng `base`, các phút không cáu trong cửa sổ bị đếm **hai lần**.

In [15]:
customers, grumpy, minutes = examples[0][:3]
base = sum(c for c, g in zip(customers, grumpy) if g == 0)
wrong = base + max(sum(customers[s:s + minutes]) for s in range(len(customers) - minutes + 1))
print("đếm đôi (sai):", wrong, "| đúng:", Solution().maxSatisfied(customers, grumpy, minutes))
assert wrong != 16

đếm đôi (sai): 23 | đúng: 16


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (cửa sổ trượt trên khách bị mất): `O(n)` thời gian, `O(1)` bộ nhớ thêm.

```python
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        n = len(customers)
        base = 0                                   # khách hài lòng sẵn (phút không cáu)
        for i in range(n):
            if grumpy[i] == 0:
                base += customers[i]
        window = 0                                 # tổng khách bị mất trong cửa sổ hiện tại
        for i in range(minutes):
            window += customers[i] * grumpy[i]
        best = window
        for i in range(minutes, n):
            window += customers[i] * grumpy[i] - customers[i - minutes] * grumpy[i - minutes]   # vào một phút, ra một phút
            best = max(best, window)
        return base + best
```

In [16]:
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        n = len(customers)
        base = 0                                   # khách hài lòng sẵn (phút không cáu)
        for i in range(n):
            if grumpy[i] == 0:
                base += customers[i]
        window = 0                                 # tổng khách bị mất trong cửa sổ hiện tại
        for i in range(minutes):
            window += customers[i] * grumpy[i]
        best = window
        for i in range(minutes, n):
            window += customers[i] * grumpy[i] - customers[i - minutes] * grumpy[i - minutes]   # vào một phút, ra một phút
            best = max(best, window)
        return base + best

sol = Solution()
for customers, grumpy, minutes, expected in examples:
    got = sol.maxSatisfied(customers, grumpy, minutes)
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

16 OK
1 OK


## 10. Vì sao đúng

**Phân tích.** Với bí kíp ở cửa sổ `W = [s, s + m)`, phút `i` hài lòng ⇔ `grumpy[i] == 0` hoặc `i` thuộc `W`. Tổng số khách hài lòng:

`sum_{i: grumpy[i]=0} customers[i] + sum_{i in W, grumpy[i]=1} customers[i] = base + sum_{i in W} lost[i]`,

vì `lost[i] = customers[i] * grumpy[i]` bằng `customers[i]` khi cáu và `0` khi không cáu (những phút không cáu trong `W` đã được tính trong `base`, và đóng góp `0` vào `lost`). Số hạng `base` không phụ thuộc `s`, nên tối đa hóa theo `s` chính là tối đa hóa tổng `lost` trong cửa sổ.

**Cửa sổ trượt.** Cửa sổ `[s + 1, s + m + 1)` có tổng bằng cửa sổ `[s, s + m)` cộng `lost[s + m]` trừ `lost[s]`. Duyệt `i = s + m` từ `m` đến `n - 1` bao phủ mọi cửa sổ (cửa sổ đầu tiên tính riêng): tối đa `n - m + 1` cửa sổ.

**Cách 2** là cùng công thức viết bằng tiền tố: `sat[s] + (tot[s + m] - tot[s]) + (sat[n] - sat[s + m])` cũng bằng `base + sum_{i in W} lost[i]` (vì `tot - sat` trong `W` chính là `lost`).

**Cách 4**: `base = sum(customers) - sum(lost)` và tổng cửa sổ bằng hiệu hai tiền tố.

**Kiểm chứng bằng máy.** Oracle (Cách 1) mô phỏng lại cả ngày cho từng vị trí bắt đầu. Ta đối chiếu ba cách còn lại trên **mọi** cấu hình nhỏ (`n <= 6`, `customers` thuộc `{0, 1, 2}`, `grumpy` nhị phân, mọi `minutes`), cộng nhiều ca ngẫu nhiên có seed (tới `n = 2 * 10^4`).

In [17]:
from itertools import product
total = 0
for n in range(1, 7):
    for cust in product((0, 1, 2), repeat=n):
        for gr in product((0, 1), repeat=n):
            for m in range(1, n + 1):
                c, g = list(cust), list(gr)
                ref = satisfied_brute(c, g, m)
                assert satisfied_prefix(c, g, m) == satisfied_window(c, g, m) == satisfied_accumulate(c, g, m) == sol.maxSatisfied(c, g, m) == ref, (c, g, m)
                total += 1
print("đã đối chiếu", total, "cấu hình nhỏ (n <= 6): các cách khớp oracle")

đã đối chiếu 324726 cấu hình nhỏ (n <= 6): các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng lại cả ngày | `O(n * (n - m + 1))` | `O(n)` |
| 2. Hai mảng tiền tố | `O(n)` | `O(n)` |
| 3. Cửa sổ trượt | `O(n)` | `O(1)` |
| 4. Tiền tố của `lost` | `O(n)` | `O(n)` |

Đo ở cỡ tối đa (`n = 2 * 10^4`) với các dạng: ngẫu nhiên, toàn cáu, không cáu, `minutes = 1`, `minutes = n`:

In [18]:
rng = random.Random(11)
n = 2 * 10 ** 4
cust = [rng.randint(0, 1000) for _ in range(n)]
cases = {"ngẫu nhiên, m = 5000": (cust, [rng.randint(0, 1) for _ in range(n)], 5000), "toàn cáu, m = 100": (cust, [1] * n, 100),
         "không cáu, m = 100": (cust, [0] * n, 100), "m = 1": (cust, [rng.randint(0, 1) for _ in range(n)], 1), "m = n": (cust, [rng.randint(0, 1) for _ in range(n)], n)}
for label, (c, g, m) in cases.items():
    ref = satisfied_window(c, g, m)
    row = [f"{label:22}"]
    for name, fn in (("tiền tố", sol_prefix), ("cửa sổ", sol_window), ("accumulate", sol_accumulate), ("nộp", sol.maxSatisfied)):
        t0 = time.perf_counter()
        got = fn(c, g, m)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:6.2f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên, m = 5000   | tiền tố   7.55 ms | cửa sổ   5.02 ms | accumulate   4.39 ms | nộp   5.52 ms | đáp án 6258372
toàn cáu, m = 100      | tiền tố   7.34 ms | cửa sổ   5.74 ms | accumulate   5.15 ms | nộp   6.05 ms | đáp án 58040
không cáu, m = 100     | tiền tố   7.38 ms | cửa sổ   5.23 ms | accumulate   3.51 ms | nộp   5.16 ms | đáp án 9938117
m = 1                  | tiền tố   8.78 ms | cửa sổ   6.23 ms | accumulate   4.98 ms | nộp   5.93 ms | đáp án 4949076
m = n                  | tiền tố   3.26 ms | cửa sổ   3.49 ms | accumulate   2.56 ms | nộp   3.48 ms | đáp án 9938117


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và các tính chất.

In [19]:
def all_solutions():
    return {"prefix": satisfied_prefix, "window": satisfied_window, "accumulate": satisfied_accumulate,
            "sol_prefix": sol_prefix, "sol_window": sol_window, "sol_accumulate": sol_accumulate, "final": sol.maxSatisfied}

def check_all(c, g, m, expected=None, brute=True):
    results = {name: fn(list(c), list(g), m) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (c[:10], g[:10], m, results)
    ref = results["window"]
    if brute and len(c) <= 200:
        assert satisfied_brute(c, g, m) == ref, (c[:10], g[:10], m)
    if expected is not None:
        assert ref == expected, (c, g, m, ref, expected)
    return ref

# 12.1 ví dụ đề
for c, g, m, expected in examples:
    check_all(c, g, m, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([5], [1], 1, 5),                                   # một phút cáu, dùng bí kíp: hài lòng 5
    ([5], [0], 1, 5),                                   # một phút không cáu
    ([0], [1], 1, 0),                                   # không có khách
    ([3, 4, 5], [1, 1, 1], 3, 12),                      # toàn cáu, bí kíp phủ cả ngày
    ([3, 4, 5], [1, 1, 1], 1, 5),                       # toàn cáu, chỉ cứu được phút đông nhất
    ([3, 4, 5], [0, 0, 0], 2, 12),                      # không cáu: tất cả hài lòng sẵn
    ([4, 10, 10], [1, 1, 0], 2, 24),                    # tính tay ở bài tập 2
    ([1, 2, 3, 4], [1, 0, 1, 0], 2, 9),                 # base = 6, lost = [1,0,3,0]: cửa sổ tốt nhất thêm 3
    ([10, 1, 1, 10], [1, 0, 0, 1], 1, 12),              # m = 1: cứu một phút cáu 10 + hài lòng sẵn 1 + 1
    ([10, 1, 1, 10], [1, 0, 0, 1], 4, 22),              # m = n: cứu hết
    ([1, 1, 1, 1, 100], [1, 1, 1, 1, 1], 2, 101),       # cửa sổ tối ưu ở cuối: 1 + 100
    ([100, 1, 1, 1, 1], [1, 1, 1, 1, 1], 2, 101),       # cửa sổ tối ưu ở đầu
    ([1000] * 5, [1] * 5, 5, 5000),                     # biên giá trị
    ([0, 0, 0], [1, 0, 1], 2, 0),                       # toàn khách 0
]
for c, g, m, expected in edge:
    check_all(c, g, m, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_subtract(c, g, m):                      # quên trừ phần tử rời cửa sổ
    base = sum(x for x, gi in zip(c, g) if gi == 0)
    lost = [x * gi for x, gi in zip(c, g)]
    window = sum(lost[:m])
    best = window
    for i in range(m, len(lost)):
        window += lost[i]
        best = max(best, window)
    return base + best

def wrong_skip_first_window(c, g, m):                # bỏ qua cửa sổ đầu tiên (best khởi tạo 0)
    base = sum(x for x, gi in zip(c, g) if gi == 0)
    lost = [x * gi for x, gi in zip(c, g)]
    window = sum(lost[:m])
    best = 0
    for i in range(m, len(lost)):
        window += lost[i] - lost[i - m]
        best = max(best, window)
    return base + best

def wrong_double_count(c, g, m):                     # tính cả khách hài lòng sẵn trong cửa sổ (đếm đôi)
    base = sum(x for x, gi in zip(c, g) if gi == 0)
    return base + max(sum(c[s:s + m]) for s in range(len(c) - m + 1))

def wrong_window_off_by_one(c, g, m):                # cửa sổ độ dài m - 1
    return satisfied_window(c, g, max(1, m - 1)) if m > 1 else satisfied_window(c, g, m)

def wrong_greedy_single_minute(c, g, m):             # tham lam: chọn phút cáu đông nhất rồi cứu cửa sổ bắt đầu từ đó
    base = sum(x for x, gi in zip(c, g) if gi == 0)
    lost = [x * gi for x, gi in zip(c, g)]
    i = max(range(len(lost)), key=lambda k: lost[k])
    s = min(i, len(c) - m)
    return base + sum(lost[s:s + m])

def wrong_no_base(c, g, m):                          # quên cộng khách hài lòng sẵn
    lost = [x * gi for x, gi in zip(c, g)]
    return max(sum(lost[s:s + m]) for s in range(len(c) - m + 1))

def find_counter(wrong, tries=30000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 7)
        c = [rng.randint(0, 5) for _ in range(n)]
        g = [rng.randint(0, 1) for _ in range(n)]
        m = rng.randint(1, n)
        if wrong(c, g, m) != satisfied_brute(c, g, m):
            return c, g, m
    return None

counters = {}
for name, fn in (("quên trừ phần tử rời cửa sổ", wrong_no_subtract), ("bỏ qua cửa sổ đầu tiên", wrong_skip_first_window),
                 ("đếm đôi khách hài lòng sẵn", wrong_double_count), ("cửa sổ lệch một (m - 1)", wrong_window_off_by_one),
                 ("tham lam một phút", wrong_greedy_single_minute), ("quên cộng base", wrong_no_base)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    c, g, m = case
    print(f"{name:34} customers={c} grumpy={g} m={m}: sai -> {fn(c, g, m)}, đúng -> {satisfied_brute(c, g, m)}")
    check_all(c, g, m)
print("12.3 phản ví dụ: OK")

quên trừ phần tử rời cửa sổ        customers=[1, 5, 3, 0, 2, 1, 3] grumpy=[1, 1, 0, 0, 0, 1, 0] m=4: sai -> 15, đúng -> 14
bỏ qua cửa sổ đầu tiên             customers=[4, 2, 2, 1, 1] grumpy=[0, 1, 1, 0, 1] m=5: sai -> 5, đúng -> 10
đếm đôi khách hài lòng sẵn         customers=[4, 2, 2, 1, 1] grumpy=[0, 1, 1, 0, 1] m=5: sai -> 15, đúng -> 10
cửa sổ lệch một (m - 1)            customers=[1, 5, 1, 0, 0, 5, 3] grumpy=[1, 0, 1, 0, 0, 1, 0] m=4: sai -> 13, đúng -> 14
tham lam một phút                  customers=[1, 5, 3, 0, 2, 1, 3] grumpy=[1, 1, 0, 0, 0, 1, 0] m=4: sai -> 13, đúng -> 14
quên cộng base                     customers=[4, 2, 2, 1, 1] grumpy=[0, 1, 1, 0, 1] m=5: sai -> 5, đúng -> 10
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: nhiều kích thước, mật độ cáu, giá trị khách; n tới 200 so oracle, n = 2*10^4 so các cách nhanh
rng = random.Random(4052)
for _ in range(600):
    n = rng.randint(1, 30)
    p = rng.choice([0.2, 0.5, 0.8])
    c = [rng.randint(0, rng.choice([2, 10, 1000])) for _ in range(n)]
    g = [1 if rng.random() < p else 0 for _ in range(n)]
    check_all(c, g, rng.randint(1, n))
for _ in range(10):
    n = rng.randint(100, 200)
    check_all([rng.randint(0, 1000) for _ in range(n)], [rng.randint(0, 1) for _ in range(n)], rng.randint(1, n))
for _ in range(5):
    n = rng.randint(15000, 20000)
    check_all([rng.randint(0, 1000) for _ in range(n)], [rng.randint(0, 1) for _ in range(n)], rng.randint(1, n), brute=False)
print("12.4 ngẫu nhiên: OK")

# 12.5 tính chất: m = n cho tổng mọi khách; không cáu thì không đổi theo m; đơn điệu tăng theo m; >= base và <= tổng; thêm khách không làm giảm
rng = random.Random(5052)
for _ in range(300):
    n = rng.randint(1, 40)
    c = [rng.randint(0, 1000) for _ in range(n)]
    g = [rng.randint(0, 1) for _ in range(n)]
    base = sum(x for x, gi in zip(c, g) if gi == 0)
    vals = [sol.maxSatisfied(c, g, m) for m in range(1, n + 1)]
    assert vals[-1] == sum(c)                                        # bí kíp phủ cả ngày
    assert vals == sorted(vals) and vals[0] >= base and vals[-1] <= sum(c)    # đơn điệu theo m
    assert sol.maxSatisfied(c, [0] * n, rng.randint(1, n)) == sum(c)
    i = rng.randrange(n)
    c2 = list(c); c2[i] += 5
    m = rng.randint(1, n)
    assert sol.maxSatisfied(c2, g, m) >= sol.maxSatisfied(c, g, m)           # thêm khách không làm giảm đáp án
print("12.5 tính chất: OK")

12.4 ngẫu nhiên: OK
12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ phần tử rời cửa sổ

Cửa sổ trượt phải **cộng phần tử mới và trừ phần tử cũ** để giữ đúng độ dài `minutes`. Chỉ cộng làm cửa sổ ngày càng dài.

In [22]:
c, g, m = counters["quên trừ phần tử rời cửa sổ"]
print("customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_no_subtract(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_no_subtract(c, g, m) != Solution().maxSatisfied(c, g, m)

customers = [1, 5, 3, 0, 2, 1, 3] grumpy = [1, 1, 0, 0, 0, 1, 0] m = 4 | sai -> 15 | đúng -> 14


Vì sao sai: tổng "cửa sổ" tăng không ngừng, vượt khả năng thực. **Đúng:** `window += lost[i] - lost[i - minutes]`.

### Lỗi 2 — Bỏ qua cửa sổ đầu tiên

Cửa sổ `[0, minutes)` cũng là một lựa chọn; khởi tạo `best = 0` rồi chỉ xét các cửa sổ sau bỏ lỡ nó.

In [23]:
c, g, m = counters["bỏ qua cửa sổ đầu tiên"]
print("customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_skip_first_window(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_skip_first_window(c, g, m) != Solution().maxSatisfied(c, g, m)

customers = [4, 2, 2, 1, 1] grumpy = [0, 1, 1, 0, 1] m = 5 | sai -> 5 | đúng -> 10


Vì sao sai: ví dụ `m = n` chỉ có một cửa sổ. **Đúng:** `best = window` ngay sau khi tính cửa sổ đầu tiên.

### Lỗi 3 — Đếm đôi khách hài lòng sẵn

Dùng tổng `customers` trong cửa sổ rồi cộng `base` làm các phút **không cáu** trong cửa sổ bị tính hai lần.

In [24]:
c, g, m = counters["đếm đôi khách hài lòng sẵn"]
print("customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_double_count(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_double_count(c, g, m) != Solution().maxSatisfied(c, g, m)

customers = [4, 2, 2, 1, 1] grumpy = [0, 1, 1, 0, 1] m = 5 | sai -> 15 | đúng -> 10


Vì sao sai: khách ở phút không cáu đã nằm trong `base`. **Đúng:** chỉ cộng `customers[i] * grumpy[i]` (khách bị mất).

### Lỗi 4 — Cửa sổ lệch một đơn vị

Bí kíp phủ **đúng** `minutes` phút liên tiếp (`[s, s + minutes)`).

In [25]:
c, g, m = counters["cửa sổ lệch một (m - 1)"]
print("customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_window_off_by_one(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_window_off_by_one(c, g, m) != Solution().maxSatisfied(c, g, m)

customers = [1, 5, 1, 0, 0, 5, 3] grumpy = [1, 0, 1, 0, 0, 1, 0] m = 4 | sai -> 13 | đúng -> 14


Vì sao sai: cửa sổ ngắn hơn cứu được ít phút hơn. **Đúng:** `range(minutes)` cho cửa sổ đầu và `i - minutes` khi trượt.

### Lỗi 5 — Tham lam một phút, hoặc quên cộng `base`

Chọn phút cáu đông nhất rồi cứu quanh đó bỏ qua các cửa sổ tốt hơn; và quên `base` làm mất khách hài lòng sẵn.

In [26]:
c, g, m = counters["tham lam một phút"]
print("tham lam: customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_greedy_single_minute(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_greedy_single_minute(c, g, m) != Solution().maxSatisfied(c, g, m)
c, g, m = counters["quên cộng base"]
print("quên base: customers =", c, "grumpy =", g, "m =", m, "| sai ->", wrong_no_base(c, g, m), "| đúng ->", Solution().maxSatisfied(c, g, m))
assert wrong_no_base(c, g, m) != Solution().maxSatisfied(c, g, m)

tham lam: customers = [1, 5, 3, 0, 2, 1, 3] grumpy = [1, 1, 0, 0, 0, 1, 0] m = 4 | sai -> 13 | đúng -> 14
quên base: customers = [4, 2, 2, 1, 1] grumpy = [0, 1, 1, 0, 1] m = 5 | sai -> 5 | đúng -> 10


Vì sao sai: cửa sổ tối ưu là cửa sổ có **tổng** khách bị mất lớn nhất, không nhất thiết chứa phút đơn lẻ đông nhất; và đáp án phải cộng khách hài lòng sẵn. **Đúng:** `base + max tổng cửa sổ`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chọn **một** đoạn liên tiếp độ dài cố định để cải thiện tối đa" ⇒ **cửa sổ trượt độ dài cố định**; tách "phần cố định" và "phần cải thiện được trong cửa sổ".
- Cửa sổ trượt: `window += new - old`; khởi tạo cửa sổ đầu, `best = window`, rồi trượt.
- Tính "phần cải thiện" riêng (`lost[i] = customers[i] * grumpy[i]`) để tránh đếm đôi.
- Tổng tiền tố cho tổng đoạn bất kỳ trong `O(1)`.
- Cùng họ: Maximum Average Subarray I (643), Maximum Points You Can Obtain from Cards (1423), Max Consecutive Ones III (1004), Sliding Window Maximum (239).
- Test nên có: `minutes = 1`, `minutes = n`, toàn cáu, không cáu, khách `0`, cửa sổ tối ưu ở đầu/cuối.

## 15. Biến thể

1. **Dùng bí kíp tối đa `k` lần** với các cửa sổ không giao nhau: quy hoạch động hoặc tham lam trên các cửa sổ tốt nhất (như bài 689).
2. **Độ dài bí kíp khác nhau.**
3. **Chỉ số cửa sổ tối ưu:** trả về vị trí bắt đầu (bài tập 5).
4. Bài liên quan: 643, 689, 1423.

Biến thể 1 với `k = 2` cửa sổ **không giao nhau** (kiểm bằng oracle vét cạn mọi cặp vị trí):

In [27]:
def best_two_windows(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    base = sum(c for c, g in zip(customers, grumpy) if g == 0)
    lost = [c * g for c, g in zip(customers, grumpy)]
    pre = list(accumulate(lost, initial=0))
    win = [pre[i + minutes] - pre[i] for i in range(n - minutes + 1)]
    best = base
    left_best = [0] * len(win)
    cur = 0
    for i, w in enumerate(win):
        cur = max(cur, w)
        left_best[i] = cur
    for j in range(len(win)):
        i = j - minutes
        add = win[j] + (left_best[i] if i >= 0 else 0)
        best = max(best, base + add)
    return best

def best_two_windows_brute(customers, grumpy, minutes):
    n = len(customers)
    starts = range(n - minutes + 1)
    def happy(chosen):
        g = grumpy[:]
        for s in chosen:
            for i in range(s, s + minutes):
                g[i] = 0
        return sum(c for c, gi in zip(customers, g) if gi == 0)
    best = max(happy([s]) for s in starts)                      # dùng một cửa sổ
    for s1 in starts:
        for s2 in starts:
            if s2 >= s1 + minutes:                              # hai cửa sổ không giao nhau
                best = max(best, happy([s1, s2]))
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 10)
    c = [rng.randint(0, 9) for _ in range(n)]
    g = [rng.randint(0, 1) for _ in range(n)]
    m = rng.randint(1, max(1, n // 2))
    assert best_two_windows(c, g, m) == best_two_windows_brute(c, g, m), (c, g, m)
print("biến thể hai cửa sổ không giao nhau: kiểm bằng oracle vét cạn")

biến thể hai cửa sổ không giao nhau: kiểm bằng oracle vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng lại cả ngày | Theo định nghĩa | `O(n^2)` | `O(n)` | Oracle |
| 2. Hai mảng tiền tố | Trước + cửa sổ + sau | `O(n)` | `O(n)` | Dễ suy ra |
| 3. Cửa sổ trượt | `window += new - old` | `O(n)` | `O(1)` | **Cách nộp chính** |
| 4. Tiền tố của `lost` | `accumulate` | `O(n)` | `O(n)` | Viết gọn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
c, g = [2, 3, 4], [1, 0, 1]
print([x * gi for x, gi in zip(c, g)], sum(x for x, gi in zip(c, g) if gi == 0))
print(list(accumulate([2, 0, 4], initial=0)), Solution().maxSatisfied(c, g, 2))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách lost và base
my_guess_2 = None      # danh sách tiền tố và đáp án

### Bài 2 — Tính tay (dễ-vừa)

`customers = [4, 10, 10]`, `grumpy = [1, 1, 0]`, `minutes = 2`. Ghi đáp án.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cửa sổ trượt.

In [30]:
def exercise_3(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    base = sum(c for c, g in zip(customers, grumpy) if g == 0)
    lost = [c * g for c, g in zip(customers, grumpy)]
    window = sum(lost[:minutes])
    best = window
    for i in range(minutes, n):
        # TODO 1: cập nhật window: cộng lost[i], trừ lost[i - minutes]
        # TODO 2: cập nhật best
        pass
    # TODO 3: trả về base + best
    pass


# for c, g, m, expected in examples:
#     assert exercise_3(c, g, m) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(customers, grumpy, minutes)` bằng **hai mảng tiền tố** (`sat` và `tot`).

In [31]:
def exercise_4(customers: List[int], grumpy: List[int], minutes: int) -> int:
    # TODO
    pass


# for c, g, m, expected in examples:
#     assert exercise_4(c, g, m) == expected

### Bài 5 — Mini challenge: vị trí bắt đầu tối ưu (khó)

Viết `exercise_5(customers, grumpy, minutes)` trả về **một** chỉ số bắt đầu `s` của bí kíp sao cho số khách hài lòng đạt tối đa. Có thể có nhiều đáp án đúng; ta kiểm tra bằng **hàm kiểm tra** (mô phỏng cả ngày với bí kíp ở `s`).

```python
exercise_5([1,0,1,2,1,1,7,5], [0,1,0,1,0,1,0,1], 3)  -> 5   (cửa sổ phút 5..7)
exercise_5([1], [0], 1)                              -> 0
```
Gợi ý: như cửa sổ trượt nhưng nhớ chỉ số cửa sổ tốt nhất.

In [32]:
def exercise_5(customers: List[int], grumpy: List[int], minutes: int) -> int:
    # TODO
    pass


def check_start(customers, grumpy, minutes, s) -> bool:
    if not (0 <= s <= len(customers) - minutes):
        return False
    g = grumpy[:]
    for i in range(s, s + minutes):
        g[i] = 0
    return sum(c for c, gi in zip(customers, g) if gi == 0) == Solution().maxSatisfied(customers, grumpy, minutes)


# for c, g, m, _ in examples:
#     assert check_start(c, g, m, exercise_5(c, g, m))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `lost[i] = c[i] * g[i]`; `base` là tổng khách ở phút không cáu.
- Mức 2: `c = [2, 3, 4]`, `g = [1, 0, 1]`: `lost = [2, 0, 4]`, `base = 3`.
- Mức 3: dòng 1: `[2, 0, 4] 3`; dòng 2: `[0, 2, 2, 6] 7` (cửa sổ `2` phút tốt nhất có `lost` `4`: `3 + 4 = 7`).

**Bài 2**
- Mức 1: `base = 10` (phút `2` không cáu).
- Mức 2: `lost = [4, 10, 0]`; cửa sổ `2` phút: `14` hoặc `10`.
- Mức 3: `10 + 14 = 24`.

**Bài 3**
- Mức 1: `window += lost[i] - lost[i - minutes]`.
- Mức 2: `best = max(best, window)`.
- Mức 3: `return base + best`.

**Bài 4**
- Mức 1: `sat = list(accumulate(..., initial=0))`, `tot = list(accumulate(customers, initial=0))`.
- Mức 2: với `s`: `sat[s] + tot[s + m] - tot[s] + sat[n] - sat[s + m]`.
- Mức 3: lấy `max` theo `s` từ `0` đến `n - m`.

**Bài 5**
- Mức 1: lưu `best_start = 0` cùng `best = window`.
- Mức 2: khi `window > best` cập nhật `best_start = i - minutes + 1`.
- Mức 3: trả `best_start`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
c, g = [2, 3, 4], [1, 0, 1]
print([x * gi for x, gi in zip(c, g)], sum(x for x, gi in zip(c, g) if gi == 0))
print(list(accumulate([2, 0, 4], initial=0)), Solution().maxSatisfied(c, g, 2))

assert ([x * gi for x, gi in zip(c, g)], sum(x for x, gi in zip(c, g) if gi == 0)) == ([2, 0, 4], 3)
assert (list(accumulate([2, 0, 4], initial=0)), Solution().maxSatisfied(c, g, 2)) == ([0, 2, 2, 6], 7)

[2, 0, 4] 3
[0, 2, 2, 6] 7


### Đáp án Bài 2

`base = 10`; `lost = [4, 10, 0]`; tổng cửa sổ `2` phút: `[14, 10]`; tốt nhất `14`; đáp án `24`.

In [34]:
hand_answer_solution = 24
assert Solution().maxSatisfied([4, 10, 10], [1, 1, 0], 2) == hand_answer_solution == satisfied_brute([4, 10, 10], [1, 1, 0], 2)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    base = sum(c for c, g in zip(customers, grumpy) if g == 0)
    lost = [c * g for c, g in zip(customers, grumpy)]
    window = sum(lost[:minutes])
    best = window
    for i in range(minutes, n):
        window += lost[i] - lost[i - minutes]
        best = max(best, window)
    return base + best

for c, g, m, expected in examples:
    assert exercise_3_solution(c, g, m) == expected
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 12)
    c = [rng.randint(0, 9) for _ in range(n)]
    g = [rng.randint(0, 1) for _ in range(n)]
    m = rng.randint(1, n)
    assert exercise_3_solution(c, g, m) == satisfied_brute(c, g, m)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(customers: List[int], grumpy: List[int], minutes: int) -> int:
    n = len(customers)
    sat = list(accumulate((c if g == 0 else 0 for c, g in zip(customers, grumpy)), initial=0))
    tot = list(accumulate(customers, initial=0))
    return max(sat[s] + tot[s + minutes] - tot[s] + sat[n] - sat[s + minutes] for s in range(n - minutes + 1))

for c, g, m, expected in examples:
    assert exercise_4_solution(c, g, m) == expected
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 12)
    c = [rng.randint(0, 9) for _ in range(n)]
    g = [rng.randint(0, 1) for _ in range(n)]
    m = rng.randint(1, n)
    assert exercise_4_solution(c, g, m) == satisfied_brute(c, g, m)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(customers: List[int], grumpy: List[int], minutes: int) -> int:
    lost = [c * g for c, g in zip(customers, grumpy)]
    window = sum(lost[:minutes])
    best, best_start = window, 0
    for i in range(minutes, len(lost)):
        window += lost[i] - lost[i - minutes]
        if window > best:
            best, best_start = window, i - minutes + 1
    return best_start

for c, g, m, _ in examples:
    assert check_start(c, g, m, exercise_5_solution(c, g, m))
assert exercise_5_solution(*examples[0][:3]) == 5
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 12)
    c = [rng.randint(0, 9) for _ in range(n)]
    g = [rng.randint(0, 1) for _ in range(n)]
    m = rng.randint(1, n)
    assert check_start(c, g, m, exercise_5_solution(c, g, m)), (c, g, m)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (cửa sổ trượt cố định)

```python
class Solution:
    def maxSatisfied(self, customers: List[int], grumpy: List[int], minutes: int) -> int:
        n = len(customers)
        base = 0                                   # khách hài lòng sẵn (phút không cáu)
        for i in range(n):
            if grumpy[i] == 0:
                base += customers[i]
        window = 0                                 # tổng khách bị mất trong cửa sổ hiện tại
        for i in range(minutes):
            window += customers[i] * grumpy[i]
        best = window
        for i in range(minutes, n):
            window += customers[i] * grumpy[i] - customers[i - minutes] * grumpy[i - minutes]
            best = max(best, window)
        return base + best
```

Công thức nhớ nhanh: đáp án `= base + max tổng (customers[i] * grumpy[i]) trên cửa sổ độ dài minutes`.

### Checklist trước khi submit

- [ ] Tách `base` (phút không cáu) và chỉ cộng khách **bị mất** (`customers * grumpy`) trong cửa sổ?
- [ ] Tính cửa sổ đầu `[0, minutes)` và gán `best = window` trước khi trượt?
- [ ] Trượt đúng: cộng phần tử mới `i`, **trừ** phần tử rời `i - minutes`?
- [ ] Trả `base + best`?
- [ ] Đã thử: `minutes = 1`, `minutes = n`, toàn cáu, không cáu, khách `0`, `n = 2 * 10^4`?